# Tier-2 High-Value Experiment: ISOMORPH T=221 Chronological Control Campaign

This notebook executes the **sample-size-controlled experiment** for ISOMORPH.
- Controls observation length strictly to **$T = 221$ time steps** (matching SupplyGraph's exact observation window).
- Uses the standard **$L=14$ lookback**, **$H=7$ horizon**, and **70% / 15% / 15% chronological split** ($154$ train / $33$ val / $34$ test steps).
- Generates **$N_{win} = 28$ test windows**, exactly matching Table I SupplyGraph.
- Evaluates 4 models: **Temporal-only**, **STGT (Real routing graph)**, **STGT_Permuted (graph_seed=0, 0 shared edges)**, and **Adaptive_STGT**.
- Runs across 5 seeds: `[42, 123, 2026, 7, 99]` (20 runs total).
- Uses exact Google Drive OAuth authentication with `drive_token.json` and `DriveStore`.
- Implements atomic checkpointing on every epoch (`epoch-XXXX-<sha>.pt`, `recovery.json`), incremental row-by-row CSV upload to Drive (`isomorph_t221_results.csv`), and automatic resumption on exit code 124.

In [ ]:
# CELL 1: Paths and Drive Setup
from pathlib import Path
import os

TOKEN_FILE = Path('/kaggle/input/datasets/mohdakifaliparvez/google-drive-oauth-token-1/drive_token.json')
DRIVE_ROOT_FOLDER_ID = '1jMUTXWHeee5LVuRralPSP1dFOLHXith-'
REPO = Path('/kaggle/working/research')
WORK_ROOT = Path('/kaggle/working/isomorph_t221_control')
SUPPORT = Path('/kaggle/working/isomorph_t221_support')
ISOMORPH_PROCESSED = Path('/kaggle/working/data/isomorph')

In [ ]:
# CELL 2: Repository Clone & Minimal Dependencies
import importlib.util
import subprocess
import sys
import torch

if not REPO.exists():
    print('Cloning research repository...')
    subprocess.run(['git', 'clone', '-b', 'main', 'https://github.com/Akifali-1/research.git', str(REPO)], check=True)

# Install missing dependencies without overriding Kaggle CUDA/PyTorch
missing = []
for package, module in [
    ('google-api-python-client', 'googleapiclient'),
    ('google-auth', 'google.oauth2'),
    ('torch-geometric', 'torch_geometric'),
    ('scipy', 'scipy'),
]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)

if missing:
    print('Installing missing packages:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *missing], check=True)

if torch.cuda.is_available():
    print('PyTorch:', torch.__version__, '| CUDA Available: True | GPU:', torch.cuda.get_device_name(0))
else:
    print('PyTorch:', torch.__version__, '| CUDA Available: False | Running on CPU runtime.')

In [ ]:
# CELL 3: DriveStore, Helpers & Dataset Sync
import sys
from pathlib import Path

HELPERS = {
    'kaggle_drive.py': '"""Google Drive API storage using the notebook\'s authorized-user OAuth token."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport socket\nimport tempfile\nimport time\nfrom pathlib import Path, PurePosixPath\n\n\nDEFAULT_TOKEN_FILE = Path("/kaggle/input/datasets/mohdakifaliparvez/google-drive-oauth-token-1/drive_token.json")\nDEFAULT_ROOT_ID = "1jMUTXWHeee5LVuRralPSP1dFOLHXith-"\n\n\ndef checksum(path, algorithm="md5"):\n    digest = hashlib.new(algorithm)\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef atomic_json(path, payload):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, name = tempfile.mkstemp(prefix=".json-", suffix=".tmp", dir=path.parent)\n    temporary = Path(name)\n    try:\n        with os.fdopen(descriptor, "w", encoding="utf-8") as handle:\n            json.dump(payload, handle, indent=2, sort_keys=True)\n            handle.flush()\n            os.fsync(handle.fileno())\n        temporary.replace(path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef find_token_file(preferred_path: Path | str | None = None) -> Path:\n    """Find the drive token file across standard Kaggle and local locations."""\n    candidates = []\n    if preferred_path:\n        candidates.append(Path(preferred_path))\n    candidates.extend([\n        DEFAULT_TOKEN_FILE,\n        Path("/kaggle/input/google-drive-oauth-token-1/drive_token.json"),\n        Path("/kaggle/input/datasets/mohdakifaliparvez/google-drive-oauth-token-1/token.json"),\n        Path("/kaggle/input/google-drive-oauth-token-1/token.json"),\n        Path("/kaggle/working/drive_token.json"),\n        Path("/kaggle/working/token.json"),\n        Path("drive_token.json"),\n        Path("token.json"),\n        Path.home() / ".gdrive" / "drive_token.json",\n        Path.home() / ".gdrive" / "token.json",\n    ])\n    for c in candidates:\n        if c.is_file():\n            return c.resolve()\n\n    # Dynamic glob search in /kaggle/input if mounted under custom name\n    if Path("/kaggle/input").exists():\n        found = (\n            list(Path("/kaggle/input").glob("**/drive_token*.json"))\n            + list(Path("/kaggle/input").glob("**/token*.json"))\n        )\n        if found:\n            return found[0].resolve()\n\n    # Fallback to Kaggle Secrets\n    try:\n        from kaggle_secrets import UserSecretsClient\n        secrets = UserSecretsClient()\n        for key in ["GDRIVE_TOKEN", "gdrive_token", "GOOGLE_DRIVE_TOKEN", "drive_token"]:\n            try:\n                val = secrets.get_secret(key)\n                if val:\n                    tmp = Path(tempfile.gettempdir()) / "drive_token.json"\n                    tmp.write_text(val, encoding="utf-8")\n                    return tmp.resolve()\n            except Exception:\n                pass\n    except Exception:\n        pass\n\n    return Path(preferred_path) if preferred_path else DEFAULT_TOKEN_FILE\n\n\ndef get_drive(token_file: Path | str | None = None):\n    # These are the exact OAuth credentials/build calls used in the reference notebook.\n    from google.oauth2.credentials import Credentials\n    from googleapiclient.discovery import build\n\n    actual_file = find_token_file(token_file)\n    if not actual_file.exists():\n        raise FileNotFoundError(f"Google Drive token file not found: {actual_file}")\n\n    credentials = Credentials.from_authorized_user_file(\n        str(actual_file), ["https://www.googleapis.com/auth/drive"]\n    )\n    return build("drive", "v3", credentials=credentials, cache_discovery=False)\n\n\ndef account_email(drive):\n    return drive.about().get(fields="user(emailAddress)").execute(num_retries=3)["user"]["emailAddress"]\n\n\nclass DriveStore:\n    """Preserve relative directories, replace named files, and verify downloads."""\n\n    def __init__(self, drive, root_id=DEFAULT_ROOT_ID):\n        self.drive = drive\n        self.root_id = root_id\n        self.folders = {"": root_id}\n\n    @staticmethod\n    def _quote(value):\n        return str(value).replace("\\\\", "\\\\\\\\").replace("\'", "\\\\\'")\n\n    @staticmethod\n    def _relative(name):\n        path = PurePosixPath(str(name))\n        if path.is_absolute() or ".." in path.parts or not path.parts:\n            raise ValueError("Drive artifact paths must be relative")\n        return path\n\n    def _find(self, name, parent_id):\n        query = f"name=\'{self._quote(name)}\' and \'{self._quote(parent_id)}\' in parents and trashed=false"\n        files = []\n        token = None\n        while True:\n            response = self.drive.files().list(\n                q=query,\n                pageToken=token,\n                pageSize=1000,\n                fields="nextPageToken,files(id,name,mimeType,size,md5Checksum)",\n                supportsAllDrives=True,\n                includeItemsFromAllDrives=True,\n            ).execute(num_retries=3)\n            files.extend(response.get("files", []))\n            token = response.get("nextPageToken")\n            if not token:\n                break\n        if len(files) > 1:\n            # Prefer non-folder if multiple entries, or pick the first\n            return files[0]\n        return files[0] if files else None\n\n    def folder(self, relative):\n        if str(relative) in ("", "."):\n            return self.root_id\n        path = self._relative(relative)\n        parent_id = self.root_id\n        for index, name in enumerate(path.parts, 1):\n            key = str(PurePosixPath(*path.parts[:index]))\n            if key not in self.folders:\n                found = self._find(name, parent_id)\n                if found:\n                    if found["mimeType"] != "application/vnd.google-apps.folder":\n                        raise ValueError(f"Drive path is not a folder: {key}")\n                    folder_id = found["id"]\n                else:\n                    folder_id = self.drive.files().create(\n                        body={"name": name, "parents": [parent_id], "mimeType": "application/vnd.google-apps.folder"},\n                        fields="id",\n                        supportsAllDrives=True,\n                    ).execute(num_retries=3)["id"]\n                self.folders[key] = folder_id\n            parent_id = self.folders[key]\n        return parent_id\n\n    def entry(self, relative, create_parent=False):\n        path = self._relative(relative)\n        parent = str(path.parent)\n        if create_parent:\n            parent_id = self.folder(parent)\n        else:\n            parent_id = self.root_id\n            for name in path.parts[:-1]:\n                found = self._find(name, parent_id)\n                if not found:\n                    return None\n                parent_id = found["id"]\n        return self._find(path.name, parent_id)\n\n    def upload(self, local_path, relative):\n        from googleapiclient.http import MediaFileUpload\n\n        local_path = Path(local_path)\n        path = self._relative(relative)\n        parent_id = self.folder(str(path.parent))\n        found = self._find(path.name, parent_id)\n        md5 = checksum(local_path)\n        if found and found.get("md5Checksum") == md5:\n            return found\n        media = MediaFileUpload(str(local_path), resumable=True, chunksize=4 * 1024 * 1024)\n        fields = "id,name,size,md5Checksum"\n        for attempt in range(3):\n            try:\n                if found:\n                    result = self.drive.files().update(\n                        fileId=found["id"], media_body=media, fields=fields, supportsAllDrives=True,\n                    ).execute(num_retries=3)\n                else:\n                    result = self.drive.files().create(\n                        body={"name": path.name, "parents": [parent_id]},\n                        media_body=media,\n                        fields=fields,\n                        supportsAllDrives=True,\n                    ).execute(num_retries=3)\n                break\n            except (TimeoutError, socket.timeout):\n                if attempt == 2:\n                    raise\n                time.sleep(1.0 * (attempt + 1))\n        if result.get("md5Checksum") != md5:\n            raise IOError(f"Drive upload checksum mismatch: {relative}")\n        return result\n\n    def put_json(self, relative, payload):\n        with tempfile.TemporaryDirectory(prefix="drive-json-") as temporary:\n            path = Path(temporary) / "payload.json"\n            atomic_json(path, payload)\n            return self.upload(path, relative)\n\n    def get_json(self, relative):\n        found = self.entry(relative)\n        if not found:\n            return None\n        content = self.drive.files().get_media(fileId=found["id"], supportsAllDrives=True).execute(num_retries=3)\n        return json.loads(content)\n\n    def download(self, relative, local_path):\n        from googleapiclient.http import MediaIoBaseDownload\n\n        found = self.entry(relative)\n        if not found:\n            raise FileNotFoundError(f"Missing Drive artifact: {relative}")\n        local_path = Path(local_path)\n        local_path.parent.mkdir(parents=True, exist_ok=True)\n        descriptor, name = tempfile.mkstemp(prefix=".download-", suffix=".tmp", dir=local_path.parent)\n        temporary = Path(name)\n        try:\n            with os.fdopen(descriptor, "wb") as handle:\n                request = self.drive.files().get_media(fileId=found["id"], supportsAllDrives=True)\n                downloader = MediaIoBaseDownload(handle, request, chunksize=4 * 1024 * 1024)\n                done = False\n                while not done:\n                    _, done = downloader.next_chunk(num_retries=3)\n                handle.flush()\n                os.fsync(handle.fileno())\n            if checksum(temporary) != found.get("md5Checksum"):\n                raise IOError(f"Drive download checksum mismatch: {relative}")\n            temporary.replace(local_path)\n        finally:\n            temporary.unlink(missing_ok=True)\n        return local_path\n\n    def delete(self, relative):\n        found = self.entry(relative)\n        if found:\n            self.drive.files().delete(fileId=found["id"], supportsAllDrives=True).execute(num_retries=3)\n',
    'kaggle_control_campaign.py': '"""Resumable multi-model control and expansion campaign runner.\n\nFollows the exact Google Drive storage, checkpoint recovery, and incremental CSV\nsynchronization pattern from the reference notebook.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom contextlib import redirect_stderr, redirect_stdout\nfrom pathlib import Path\n\nfrom kaggle_drive import DriveStore, account_email, atomic_json, checksum, find_token_file, get_drive\n\nROW_FIELDS = (\n    "dataset",\n    "model",\n    "seed",\n    "MAE",\n    "RMSE",\n    "WAPE",\n    "sMAPE",\n    "R2",\n    "MAPE_masked",\n    "MAPE_valid_fraction",\n    "MSE",\n    "epochs_completed",\n    "selected_epoch",\n    "parameter_count",\n    "training_seconds",\n)\n\nARTIFACT_FILES = (\n    "config.yaml",\n    "runtime.json",\n    "processed_manifest.json",\n    "training.log",\n    "checkpoints/best.pt",\n    "checkpoints/latest.pt",\n    "predictions/validation.npz",\n    "metrics/validation.json",\n    "logs/history.json",\n)\n\n\ndef digest(path, algorithm="sha256"):\n    hasher = hashlib.new(algorithm)\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            hasher.update(block)\n    return hasher.hexdigest()\n\n\ndef manifest(root, names):\n    result = {}\n    for name in names:\n        p = Path(root) / name\n        if p.exists():\n            result[name] = {\n                "size": p.stat().st_size,\n                "sha256": digest(p),\n                "md5Checksum": digest(p, "md5"),\n            }\n    return result\n\n\nclass Tee:\n    def __init__(self, console, file):\n        self.console, self.file = console, file\n\n    def write(self, text):\n        self.console.write(text)\n        self.file.write(text)\n        self.flush()\n        return len(text)\n\n    def flush(self):\n        self.console.flush()\n        self.file.flush()\n\n\nclass Persistence:\n    """Manages incremental per-epoch checkpoints and final artifact sync to Google Drive."""\n\n    def __init__(self, store: DriveStore, root: Path, run_id: str, signature: str, on_epoch):\n        self.store, self.root, self.run_id = store, Path(root), run_id\n        self.signature, self.on_epoch = signature, on_epoch\n\n    def upload(self, relative: str):\n        local_file = self.root / relative\n        if local_file.exists():\n            return self.store.upload(local_file, f"{self.run_id}/{relative}")\n        return None\n\n    def epoch(self, epoch: int):\n        checkpoint = self.root / "checkpoints/latest.pt"\n        if not checkpoint.exists():\n            return\n        sha = digest(checkpoint)\n        relative = f"{self.run_id}/checkpoints/epoch-{epoch:04d}-{sha[:12]}.pt"\n        self.upload("checkpoints/best.pt")\n        self.upload("logs/history.json")\n        self.upload("training.log")\n        remote = self.store.upload(checkpoint, relative)\n        pointer = {\n            "experiment_id": self.run_id,\n            "signature": self.signature,\n            "epoch": epoch,\n            "checkpoint": relative,\n            "sha256": sha,\n            "md5Checksum": remote.get("md5Checksum"),\n        }\n        self.store.put_json(f"{self.run_id}/recovery.json", pointer)\n        self.on_epoch(epoch)\n\n    def finish(self):\n        for relative in ARTIFACT_FILES:\n            self.upload(relative)\n        complete = {\n            "experiment_id": self.run_id,\n            "signature": self.signature,\n            "files": manifest(self.root, ARTIFACT_FILES),\n        }\n        self.store.put_json(f"{self.run_id}/complete.json", complete)\n\n\ndef hook_save_json(trainer, persistence: Persistence):\n    original = trainer.save_json\n\n    def saved(payload, path):\n        result = original(payload, path)\n        try:\n            if Path(path).resolve() == (persistence.root / "logs/history.json").resolve() and payload.get("history"):\n                # Preserve RNG state during upload\n                rng = trainer.capture_rng_state()\n                try:\n                    last_epoch = int(payload["history"][-1]["epoch"])\n                    persistence.epoch(last_epoch)\n                finally:\n                    trainer.restore_rng_state(rng)\n        except Exception as e:\n            print(f"[Persistence Warning] Non-critical error during epoch upload: {e}", flush=True)\n        return result\n\n    trainer.save_json = saved\n    return original\n\n\ndef restore(store: DriveStore, run_id: str, signature: str, local_root: Path) -> Path | None:\n    pointer = store.get_json(f"{run_id}/recovery.json")\n    if not pointer:\n        return None\n    if pointer["experiment_id"] != run_id or pointer["signature"] != signature:\n        print(f"[Warning] Recovery pointer signature mismatch for {run_id}. Ignoring stale checkpoint.", flush=True)\n        return None\n    local = store.download(pointer["checkpoint"], Path(local_root) / "checkpoints/latest.pt")\n    if digest(local) != pointer["sha256"]:\n        raise IOError(f"Recovery checkpoint sha256 mismatch for {run_id}")\n    return local\n\n\ndef load_registry(store: DriveStore, name: str, campaign: str) -> dict:\n    return store.get_json(name) or {"schema_version": 1, "campaign": campaign, "runs": {}, "updated_at": None}\n\n\ndef save_registry(store: DriveStore, work: Path, name: str, state: dict):\n    state["updated_at"] = __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat()\n    local = Path(work) / name\n    atomic_json(local, state)\n    store.upload(local, name)\n\n\ndef append_row(path: Path, row: dict):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    exists = path.exists()\n    with path.open("a", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=ROW_FIELDS)\n        if not exists:\n            writer.writeheader()\n        writer.writerow({field: row.get(field, "") for field in ROW_FIELDS})\n        handle.flush()\n\n\ndef run(args):\n    import yaml\n\n    root = args.project_root.resolve()\n    work = args.work_root.resolve()\n    processed = args.processed_root.resolve()\n    work.mkdir(parents=True, exist_ok=True)\n    sys.path.insert(0, str(root / "src"))\n    import train as trainer\n\n    base = yaml.safe_load(args.config.read_text(encoding="utf-8"))\n    base.setdefault("data", {})["dataset_name"] = args.dataset\n    if args.max_time_steps is not None:\n        base["data"]["max_time_steps"] = int(args.max_time_steps)\n    if args.max_job_seconds is not None:\n        base.setdefault("experiment", {})["max_job_seconds"] = int(args.max_job_seconds)\n\n    try:\n        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()\n    except Exception:\n        commit = "local"\n\n    token_path = find_token_file(args.token_file)\n    drive = get_drive(token_path)\n    print(f"Authenticated Drive account: {account_email(drive)}", flush=True)\n\n    root_store = DriveStore(drive, args.drive_root_folder_id)\n    campaign_id = root_store.folder(f"experiments/{args.campaign}")\n    store = DriveStore(drive, campaign_id)\n\n    registry_name = f"{args.campaign}.json"\n    state = load_registry(store, registry_name, args.campaign)\n\n    csv_name = args.results_name or f"{args.campaign}_results.csv"\n    rows_path = work / csv_name\n    if rows_path.exists():\n        rows_path.unlink()\n    remote_rows = store.entry(csv_name)\n    if remote_rows:\n        store.download(csv_name, rows_path)\n        print(f"Downloaded existing results CSV from Drive: {csv_name}", flush=True)\n\n    processed_manifest = manifest(\n        processed,\n        ["nodes.csv", "edges.csv", "sales.csv", "target_definition.json", "data_quality.json"],\n    )\n    for name in processed_manifest:\n        store.upload(processed / name, f"provenance/processed/{name}")\n    store.upload(args.config, "provenance/config.yaml")\n    save_registry(store, work, registry_name, state)\n\n    models_to_run = args.models if args.models else [args.model]\n\n    for model in models_to_run:\n        for seed in args.seeds:\n            run_id = f"{args.dataset}_{model}_seed{seed}"\n            payload = {\n                "dataset": args.dataset,\n                "model": model,\n                "seed": seed,\n                "max_time_steps": base["data"].get("max_time_steps"),\n                "config": base,\n                "processed": processed_manifest,\n            }\n            signature = hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()\n            record = state["runs"].get(run_id)\n\n            if record and record.get("status") == "completed":\n                print(f"Skipping recorded completed run: {run_id}", flush=True)\n                continue\n\n            artifact = work / run_id\n            artifact.mkdir(parents=True, exist_ok=True)\n            resume = restore(store, run_id, signature, artifact)\n            if record and resume is None and not args.restart_uncommitted:\n                print(f"[Notice] {run_id} marked as running/interrupted without checkpoint, restarting...", flush=True)\n\n            if record is None:\n                record = {\n                    "experiment_id": run_id,\n                    "signature": signature,\n                    "status": "reserved",\n                    "seed": seed,\n                    "model": model,\n                    "created_at": str(__import__("datetime").datetime.now()),\n                }\n                state["runs"][run_id] = record\n\n            config = copy.deepcopy(base)\n            config["seed"] = seed\n            config["data"]["processed_dir"] = str(processed)\n            config["outputs"] = {\n                "checkpoint_dir": str(artifact / "checkpoints"),\n                "metrics_dir": str(artifact / "metrics"),\n                "predictions_dir": str(artifact / "predictions"),\n                "logs_dir": str(artifact / "logs"),\n            }\n            config["run"] = {"experiment_id": run_id, "signature": signature, "git_commit": commit}\n\n            # Handle permuted model configuration\n            if model == "stgt_permuted":\n                config["model"] = "stgt_permuted"\n                config["data"]["graph_mode"] = "permuted"\n                config["data"]["graph_seed"] = 0\n                if "stgt_permuted" in config.get("models", {}):\n                    config["models"]["stgt_permuted"].pop("graph_seed", None)\n            else:\n                config["model"] = model\n\n            config_path = artifact / "config.yaml"\n            config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")\n            atomic_json(artifact / "processed_manifest.json", processed_manifest)\n            atomic_json(\n                artifact / "runtime.json",\n                {\n                    "git_commit": commit,\n                    "packages": {\n                        pkg: importlib.metadata.version(pkg)\n                        for pkg in ("torch", "torch-geometric", "numpy", "pandas")\n                        if importlib.util.find_spec(pkg.replace("-", "_")) is not None\n                    },\n                },\n            )\n\n            record.setdefault("sessions", []).append({"seed": seed, "resume": bool(resume)})\n            record["status"] = "running"\n            save_registry(store, work, registry_name, state)\n\n            persistence = None\n            original_save_json = None\n            try:\n                # Validation check on window counts\n                data = trainer.ExperimentData(config, root)\n                is_t221 = base["data"].get("max_time_steps") == 221 or args.dataset == "supplygraph"\n                if is_t221:\n                    expected_val_windows = 27\n                    expected_test_windows = 28\n                else:\n                    expected_val_windows = 7878\n                    expected_test_windows = 7878\n\n                val_len = len(data.datasets["validation"])\n                test_len = len(data.datasets["test"])\n                if val_len != expected_val_windows or test_len != expected_test_windows:\n                    raise RuntimeError(\n                        f"Window count mismatch for {args.dataset}: "\n                        f"validation={val_len} (expected {expected_val_windows}), "\n                        f"test={test_len} (expected {expected_test_windows})"\n                    )\n\n                def record_epoch(epoch):\n                    record["last_drive_epoch"] = epoch\n                    save_registry(store, work, registry_name, state)\n\n                persistence = Persistence(store, artifact, run_id, signature, record_epoch)\n                for rel in ("config.yaml", "runtime.json", "processed_manifest.json"):\n                    persistence.upload(rel)\n\n                original_save_json = hook_save_json(trainer, persistence)\n                log_path = artifact / "training.log"\n\n                print(f"\\n================ Starting Run: {run_id} ================", flush=True)\n                with log_path.open("a", encoding="utf-8", buffering=1) as log:\n                    with redirect_stdout(Tee(sys.stdout, log)), redirect_stderr(Tee(sys.stderr, log)):\n                        report = trainer.train_model(model, config, root, resume)\n\n                persistence.finish()\n\n                row = {\n                    "dataset": args.dataset,\n                    "model": model,\n                    "seed": seed,\n                    **report["validation"],\n                    "epochs_completed": report["epochs_completed"],\n                    "selected_epoch": report["selected_epoch"],\n                    "parameter_count": report["parameter_count"],\n                    "training_seconds": report["training_seconds"],\n                }\n                append_row(rows_path, row)\n                store.upload(rows_path, csv_name)\n                record.update(status="completed", report=row)\n                save_registry(store, work, registry_name, state)\n                print(f"Successfully completed {run_id}; result appended and synced to {csv_name}", flush=True)\n\n            except TimeoutError as exc:\n                print(\n                    f"\\n[Interrupted/Timeout] {run_id}: {type(exc).__name__}: {exc}. "\n                    f"Preserving committed epoch {record.get(\'last_drive_epoch\')} and returning code 124 for resumption.",\n                    flush=True,\n                )\n                record.update(\n                    status="interrupted",\n                    failure=f"{type(exc).__name__}: {exc}",\n                    last_drive_epoch=record.get("last_drive_epoch"),\n                )\n                try:\n                    if persistence:\n                        persistence.upload("training.log")\n                    save_registry(store, work, registry_name, state)\n                except Exception as sync_err:\n                    print(f"[Warning] Sync on interruption encountered error: {sync_err}", flush=True)\n                return 124\n\n            except BaseException as exc:\n                print(f"\\n[Error] {run_id} failed with {type(exc).__name__}: {exc}", flush=True)\n                record.update(status="failed", failure={"type": type(exc).__name__, "reason": str(exc)})\n                try:\n                    if persistence and (artifact / "training.log").exists():\n                        persistence.upload("training.log")\n                    save_registry(store, work, registry_name, state)\n                except Exception:\n                    pass\n                raise\n\n            finally:\n                if original_save_json is not None:\n                    trainer.save_json = original_save_json\n\n    print(f"\\nCompleted campaign: {args.campaign}", flush=True)\n    return 0\n\n\ndef parser():\n    p = argparse.ArgumentParser(description="Kaggle Control and Expansion Campaign Runner")\n    p.add_argument("--project-root", type=Path, default=Path("."))\n    p.add_argument("--config", type=Path, required=True)\n    p.add_argument("--processed-root", type=Path, required=True)\n    p.add_argument("--token-file", type=Path, default=None)\n    p.add_argument("--drive-root-folder-id", default="1jMUTXWHeee5LVuRralPSP1dFOLHXith-")\n    p.add_argument("--work-root", type=Path, required=True)\n    p.add_argument("--campaign", required=True)\n    p.add_argument("--dataset", required=True, choices=["isomorph", "supplygraph"])\n    p.add_argument(\n        "--model",\n        default="stgt",\n        choices=["stgt", "stgt_permuted", "temporal_only", "spatial_only", "adaptive_stgt"],\n    )\n    p.add_argument(\n        "--models",\n        nargs="+",\n        choices=["stgt", "stgt_permuted", "temporal_only", "spatial_only", "adaptive_stgt"],\n        default=None,\n    )\n    p.add_argument("--seeds", type=int, nargs="+", required=True)\n    p.add_argument("--max-time-steps", "--max-steps", dest="max_time_steps", type=int, default=None)\n    p.add_argument("--results-name", default=None)\n    p.add_argument("--max-job-seconds", type=int, default=7200)\n    p.add_argument("--restart-uncommitted", action="store_true")\n    p.add_argument("--allow-execution", action="store_true")\n    return p\n\n\nif __name__ == "__main__":\n    args = parser().parse_args()\n    if not args.allow_execution:\n        raise SystemExit("Add --allow-execution to run the campaign")\n    raise SystemExit(run(args))\n',
    'utils.py': '"""Shared configuration, reproducibility, graph, scaling, and I/O helpers."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nfrom pathlib import Path\nfrom typing import Any, Dict, Mapping, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport yaml\nfrom sklearn.preprocessing import RobustScaler, StandardScaler\n\n\ndef load_yaml(path: Path | str) -> Dict[str, Any]:\n    with Path(path).open("r", encoding="utf-8") as handle:\n        return yaml.safe_load(handle) or {}\n\n\ndef save_json(payload: Mapping[str, Any], path: Path | str) -> None:\n    output = Path(path)\n    output.parent.mkdir(parents=True, exist_ok=True)\n\n    def default(value: Any) -> Any:\n        if isinstance(value, (np.integer, np.floating)):\n            return value.item()\n        if isinstance(value, np.ndarray):\n            return value.tolist()\n        if isinstance(value, Path):\n            return str(value)\n        if isinstance(value, torch.Tensor):\n            return value.detach().cpu().tolist()\n        raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")\n\n    with output.open("w", encoding="utf-8") as handle:\n        json.dump(payload, handle, indent=2, default=default)\n\n\ndef set_seed(seed: int, deterministic: bool = True) -> None:\n    """Set all practical random seeds without claiming perfect GPU determinism."""\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    if deterministic:\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n        torch.use_deterministic_algorithms(True, warn_only=True)\n\n\ndef model_parameter_count(model: torch.nn.Module) -> int:\n    return sum(parameter.numel() for parameter in model.parameters())\n\n\ndef read_processed_data(data_dir: Path | str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    directory = Path(data_dir)\n    nodes = pd.read_csv(directory / "nodes.csv")\n    edges = pd.read_csv(directory / "edges.csv")\n    sales = pd.read_csv(directory / "sales.csv", parse_dates=["Date"])\n\n    required_nodes = {"node_id", "node_type"}\n    required_edges = {"source", "target"}\n    if not required_nodes.issubset(nodes.columns):\n        raise ValueError(f"nodes.csv must contain {sorted(required_nodes)}")\n    if not required_edges.issubset(edges.columns):\n        raise ValueError(f"edges.csv must contain {sorted(required_edges)}")\n    if "Date" not in sales.columns:\n        raise ValueError("sales.csv must contain a Date column")\n    if sales["Date"].duplicated().any():\n        raise ValueError("sales.csv contains duplicate dates")\n    if not sales["Date"].is_monotonic_increasing:\n        raise ValueError("sales.csv must be sorted in chronological order")\n    if sales["Date"].isna().any():\n        raise ValueError("sales.csv has invalid dates")\n    if not sales["Date"].diff().dropna().eq(pd.Timedelta(days=1)).all():\n        raise ValueError("sales.csv must have a continuous daily timeline")\n\n    node_ids = nodes["node_id"].astype(str).tolist()\n    if len(node_ids) != len(set(node_ids)):\n        raise ValueError("nodes.csv contains duplicate node_id values")\n    missing_sales = sorted(set(node_ids) - set(sales.columns))\n    if missing_sales:\n        raise ValueError(f"Nodes missing from sales.csv columns: {missing_sales[:10]}")\n    unknown_sales = sorted(set(sales.columns) - {"Date", *node_ids})\n    if unknown_sales:\n        raise ValueError(f"sales.csv contains columns not present in nodes.csv: {unknown_sales[:10]}")\n    values = sales[node_ids].to_numpy(dtype=np.float64)\n    if not np.isfinite(values).all() or (values < 0).any():\n        raise ValueError("Processed demand must be finite and nonnegative")\n    known = set(node_ids)\n    if not set(edges["source"].astype(str)).issubset(known):\n        raise ValueError("edges.csv contains an unknown source node")\n    if not set(edges["target"].astype(str)).issubset(known):\n        raise ValueError("edges.csv contains an unknown target node")\n    if (edges["source"].astype(str) == edges["target"].astype(str)).any():\n        raise ValueError("edges.csv contains self-loops; self-loops are added by the model if needed")\n    return nodes, edges, sales\n\n\ndef build_edge_index(\n    nodes: pd.DataFrame,\n    edges: pd.DataFrame,\n    undirected: bool = True,\n) -> torch.Tensor:\n    """Build a validated, de-duplicated edge index from processed CSV files."""\n    index = {str(node_id): i for i, node_id in enumerate(nodes["node_id"].astype(str))}\n    pairs = set()\n    for row in edges.itertuples(index=False):\n        source = str(getattr(row, "source"))\n        target = str(getattr(row, "target"))\n        if source == target:\n            continue\n        if source not in index or target not in index:\n            raise ValueError(f"Unknown edge endpoint: {source}, {target}")\n        pairs.add((index[source], index[target]))\n        if undirected:\n            pairs.add((index[target], index[source]))\n    if not pairs:\n        raise ValueError("No valid graph edges were found")\n    ordered = sorted(pairs)\n    return torch.tensor(ordered, dtype=torch.long).t().contiguous()\n\n\ndef node_type_ids(nodes: pd.DataFrame) -> torch.Tensor:\n    mapping = {"region": 0, "state": 0, "city": 1, "store": 2, "family": 3}\n    values = []\n    for value in nodes["node_type"].astype(str):\n        key = value.strip().lower()\n        if key not in mapping:\n            raise ValueError(f"Unsupported node_type {value!r}")\n        values.append(mapping[key])\n    return torch.tensor(values, dtype=torch.long)\n\n\ndef repeat_edge_index(edge_index: torch.Tensor, batch_size: int, num_nodes: int) -> torch.Tensor:\n    """Repeat a single graph for a dense batch of independent graph windows."""\n    if batch_size == 1:\n        return edge_index\n    pieces = []\n    for batch_index in range(batch_size):\n        pieces.append(edge_index + batch_index * num_nodes)\n    return torch.cat(pieces, dim=1)\n\n\ndef split_target_ranges(\n    num_steps: int,\n    lookback: int,\n    horizon: int,\n    train_ratio: float = 0.70,\n    validation_ratio: float = 0.15,\n) -> Dict[str, Tuple[int, int]]:\n    """Return target-index ranges with historical context crossing split boundaries."""\n    if lookback < 1 or horizon < 1:\n        raise ValueError("lookback and horizon must be positive")\n    if train_ratio <= 0 or validation_ratio <= 0 or train_ratio + validation_ratio >= 1:\n        raise ValueError("Train and validation ratios must be positive and leave test data")\n    train_end = int(num_steps * train_ratio)\n    validation_end = train_end + int(num_steps * validation_ratio)\n    if train_end <= lookback + horizon - 1:\n        raise ValueError("Training split is too short for the requested lookback and horizon")\n    if validation_end - train_end < horizon or num_steps - validation_end < horizon:\n        raise ValueError("Validation or test split is too short for the requested horizon")\n    return {\n        "train": (lookback, train_end - horizon + 1),\n        "validation": (train_end, validation_end - horizon + 1),\n        "test": (validation_end, num_steps - horizon + 1),\n        "train_end": (train_end, train_end),\n        "validation_end": (validation_end, validation_end),\n    }\n\n\ndef fit_scale_matrix(\n    series: np.ndarray,\n    train_end: int,\n    kind: str = "standard",\n) -> Tuple[np.ndarray, Dict[str, Any]]:\n    """Fit one scaler per node using only the training prefix."""\n    if kind not in {"standard", "robust"}:\n        raise ValueError("scaler kind must be \'standard\' or \'robust\'")\n    scaled = np.empty_like(series, dtype=np.float32)\n    state: Dict[str, Any] = {"kind": kind, "center": [], "scale": []}\n    for node_index in range(series.shape[0]):\n        values = series[node_index].astype(np.float64).reshape(-1, 1)\n        scaler = StandardScaler() if kind == "standard" else RobustScaler()\n        scaler.fit(values[:train_end])\n        transformed = scaler.transform(values).reshape(-1)\n        scaled[node_index] = transformed.astype(np.float32)\n        center = getattr(scaler, "mean_", getattr(scaler, "center_", np.array([0.0])))\n        scale = getattr(scaler, "scale_", np.array([1.0]))\n        state["center"].append(float(center[0]))\n        state["scale"].append(float(scale[0]) if float(scale[0]) != 0 else 1.0)\n    return scaled, state\n\n\ndef inverse_scale(values: np.ndarray, state: Mapping[str, Any]) -> np.ndarray:\n    center = np.asarray(state["center"], dtype=np.float64)\n    scale = np.asarray(state["scale"], dtype=np.float64)\n    return values * scale.reshape(1, -1, 1) + center.reshape(1, -1, 1)\n\n\ndef scale_values(values: np.ndarray, state: Mapping[str, Any]) -> np.ndarray:\n    center = np.asarray(state["center"], dtype=np.float64)\n    scale = np.asarray(state["scale"], dtype=np.float64)\n    return ((values - center.reshape(-1, 1)) / scale.reshape(-1, 1)).astype(np.float32)\n\n\ndef permute_edge_index(\n    edge_index: torch.Tensor,\n    num_nodes: int,\n    graph_seed: int = 0,\n) -> torch.Tensor:\n    """Construct a permuted edge index with the same edge count and 0 shared directed edges with original."""\n    real_pairs = set(zip(edge_index[0].tolist(), edge_index[1].tolist()))\n    num_edges = edge_index.shape[1]\n    rng = np.random.default_rng(graph_seed)\n\n    # 1. First attempt: node permutation with rejection (preserves degree structure where possible)\n    for _ in range(1000):\n        p = rng.permutation(num_nodes)\n        candidate = [(int(p[u]), int(p[v])) for u, v in zip(edge_index[0].tolist(), edge_index[1].tolist())]\n        if any(u == v for u, v in candidate):\n            continue\n        if len(set(candidate)) != num_edges:\n            continue\n        if any(pair in real_pairs for pair in candidate):\n            continue\n        ordered = sorted(candidate)\n        return torch.tensor(ordered, dtype=torch.long).t().contiguous()\n\n    # 2. Second attempt: sample distinct non-self directed edges with 0 overlap\n    all_possible = [(u, v) for u in range(num_nodes) for v in range(num_nodes) if u != v and (u, v) not in real_pairs]\n    if len(all_possible) < num_edges:\n        raise ValueError(\n            f"Not enough non-overlapping edges available to construct permuted graph: {len(all_possible)} < {num_edges}"\n        )\n    chosen_indices = rng.choice(len(all_possible), size=num_edges, replace=False)\n    chosen = sorted([all_possible[i] for i in chosen_indices])\n    return torch.tensor(chosen, dtype=torch.long).t().contiguous()\n\n',
    'train.py': '"""Train one model under the shared chronological experiment protocol.\n\nThis script performs real training only when explicitly invoked by the user in\nColab. Importing the module does not load data or start computation.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport time\nfrom pathlib import Path\nfrom typing import Any, Dict, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom torch.optim.lr_scheduler import ReduceLROnPlateau\n\nfrom dataset import GraphWindowDataset, make_loader\nfrom metrics import forecasting_metrics, metrics_by_horizon, metrics_by_node_type\nfrom stgt import AdaptiveSTGT, ReferenceSTGT\nfrom day2_ablation import DAY2_VARIANTS, FixedFusionSTGT, SpatialOnlySTGT, TemporalOnlySTGT\nfrom utils import (\n    build_edge_index,\n    fit_scale_matrix,\n    inverse_scale,\n    load_yaml,\n    model_parameter_count,\n    node_type_ids,\n    permute_edge_index,\n    read_processed_data,\n    save_json,\n    set_seed,\n    split_target_ranges,\n)\nfrom checkpointing import atomic_torch_save, capture_rng_state, restore_rng_state\n\n\nclass ExperimentData:\n    """Load and scale processed data once for either model."""\n\n    def __init__(self, config: Dict[str, Any], project_root: Path):\n        data_config = config["data"]\n        data_dir = Path(data_config["processed_dir"])\n        if not data_dir.is_absolute():\n            data_dir = project_root / data_dir\n        self.nodes, self.edges, self.sales = read_processed_data(data_dir)\n        self.node_names = self.nodes["node_id"].astype(str).tolist()\n        self.node_type_ids = node_type_ids(self.nodes)\n        configured_edge_types = data_config.get("graph_edge_types")\n        if configured_edge_types is not None:\n            configured_edge_types = {str(edge_type) for edge_type in configured_edge_types}\n            if "edge_type" not in self.edges.columns:\n                raise ValueError("graph_edge_types requires edges.csv to contain edge_type")\n            unknown_edge_types = configured_edge_types - set(self.edges["edge_type"].astype(str))\n            if unknown_edge_types:\n                raise ValueError(f"Configured graph edge types are absent from edges.csv: {sorted(unknown_edge_types)}")\n            self.edges = self.edges[self.edges["edge_type"].astype(str).isin(configured_edge_types)].copy()\n            if self.edges.empty:\n                raise ValueError("Configured graph edge types produced an empty graph")\n        self.edge_index = build_edge_index(self.nodes, self.edges, data_config.get("undirected_graph", True))\n        is_permuted = (\n            config.get("model") == "stgt_permuted"\n            or config.get("graph_variant") == "permuted"\n            or (data_config.get("graph_mode") == "permuted")\n        )\n        if is_permuted:\n            graph_seed = data_config.get("graph_seed", config.get("graph_seed", 0))\n            self.edge_index = permute_edge_index(self.edge_index, len(self.nodes), graph_seed=int(graph_seed))\n        self.dates = pd.to_datetime(self.sales["Date"]).dt.strftime("%Y-%m-%d").tolist()\n        raw_series = self.sales[self.node_names].to_numpy(dtype=np.float32).T\n        max_time_steps = data_config.get("max_time_steps") or data_config.get("max_steps") or config.get("max_time_steps") or config.get("max_steps")\n        if max_time_steps is not None:\n            max_time_steps = int(max_time_steps)\n            raw_series = raw_series[:, :max_time_steps]\n            self.dates = self.dates[:max_time_steps]\n        self.num_nodes, self.num_steps = raw_series.shape\n        self.lookback = int(data_config["lookback"])\n        self.horizon = int(data_config["horizon"])\n        self.ranges = split_target_ranges(\n            self.num_steps,\n            self.lookback,\n            self.horizon,\n            float(data_config.get("train_ratio", 0.70)),\n            float(data_config.get("validation_ratio", 0.15)),\n        )\n        self.train_end = self.ranges["train_end"][0]\n        self.scaled_series, self.scaler_state = fit_scale_matrix(\n            raw_series, self.train_end, str(data_config.get("scaler", "standard"))\n        )\n        self.baseline_scaled = None\n        if bool(data_config.get("use_residual_baseline", False)):\n            if self.horizon != 1:\n                raise ValueError("The causal residual adapter currently supports horizon=1 only")\n            baseline = np.zeros_like(raw_series, dtype=np.float32)\n            baseline[:, 0] = raw_series[:, 0]\n            alpha = float(data_config.get("baseline_alpha", 2.0 / 8.0))\n            for step in range(1, self.num_steps):\n                baseline[:, step] = alpha * raw_series[:, step - 1] + (1 - alpha) * baseline[:, step - 1]\n            # Use the demand scaler so residuals and predictions invert consistently.\n            center = np.asarray(self.scaler_state["center"]).reshape(-1, 1)\n            scale = np.asarray(self.scaler_state["scale"]).reshape(-1, 1)\n            self.baseline_scaled = ((baseline - center) / scale).astype(np.float32)\n\n        self.datasets = {\n            name: GraphWindowDataset(\n                self.scaled_series,\n                self.ranges[name],\n                self.lookback,\n                self.horizon,\n                self.baseline_scaled,\n            )\n            for name in ("train", "validation", "test")\n        }\n\n    def loaders(self, batch_size: int):\n        return {\n            "train": make_loader(self.datasets["train"], batch_size, shuffle=True),\n            "validation": make_loader(self.datasets["validation"], batch_size, shuffle=False),\n            "test": make_loader(self.datasets["test"], batch_size, shuffle=False),\n        }\n\n\ndef build_model(model_name: str, config: Dict[str, Any], data: ExperimentData) -> torch.nn.Module:\n    model_cfg = config["models"].get(model_name)\n    if model_cfg is None and model_name == "stgt_permuted":\n        model_cfg = config["models"].get("stgt", {})\n    model_config = dict(model_cfg or {})\n    for non_model_key in ("graph_seed", "graph_mode", "graph_variant"):\n        model_config.pop(non_model_key, None)\n    common = {"lookback": data.lookback, "horizon": data.horizon}\n    if model_name in {"stgt", "stgt_permuted"}:\n        return ReferenceSTGT(**common, **model_config)\n    if model_name == "adaptive_stgt":\n        return AdaptiveSTGT(**common, **model_config)\n    if model_name == "temporal_only":\n        return TemporalOnlySTGT(**common, **model_config)\n    if model_name == "spatial_only":\n        return SpatialOnlySTGT(**common, **model_config)\n    if model_name == "fixed_fusion":\n        return FixedFusionSTGT(**common, **model_config)\n    raise ValueError(f"Unsupported model: {model_name}")\n\n\ndef _call_model(model_name: str, model: torch.nn.Module, batch: Dict[str, torch.Tensor], edge_index, node_types):\n    if model_name in {"stgt", "stgt_permuted", "adaptive_stgt", *DAY2_VARIANTS}:\n        return model(batch["x"], edge_index, node_types)\n    return model(batch["x"], edge_index)\n\n\ndef _absolute_arrays(\n    prediction: torch.Tensor,\n    target: torch.Tensor,\n    batch: Dict[str, torch.Tensor],\n    scaler_state: Dict[str, Any],\n) -> Tuple[np.ndarray, np.ndarray]:\n    prediction_np = prediction.detach().cpu().numpy()\n    target_np = target.detach().cpu().numpy()\n    if "baseline_y" in batch:\n        baseline_np = batch["baseline_y"].detach().cpu().numpy()\n        prediction_np = prediction_np + baseline_np\n        target_np = target_np + baseline_np\n    return inverse_scale(target_np, scaler_state), inverse_scale(prediction_np, scaler_state)\n\n\n@torch.no_grad()\ndef evaluate_loader(\n    model_name: str,\n    model: torch.nn.Module,\n    loader,\n    data: ExperimentData,\n    device: torch.device,\n) -> Dict[str, Any]:\n    model.eval()\n    true_batches = []\n    prediction_batches = []\n    target_indices = []\n    edge_index = data.edge_index.to(device)\n    types = data.node_type_ids.to(device)\n    inference_seconds = 0.0\n    for batch in loader:\n        batch = {key: value.to(device) for key, value in batch.items()}\n        if device.type == "cuda":\n            torch.cuda.synchronize()\n        started = time.perf_counter()\n        prediction = _call_model(model_name, model, batch, edge_index, types)\n        if device.type == "cuda":\n            torch.cuda.synchronize()\n        inference_seconds += time.perf_counter() - started\n        true_np, prediction_np = _absolute_arrays(prediction, batch["y"], batch, data.scaler_state)\n        true_batches.append(true_np)\n        prediction_batches.append(prediction_np)\n        target_indices.extend(batch["target_index"].cpu().tolist())\n    if not true_batches:\n        raise RuntimeError("Evaluation loader is empty")\n    y_true = np.concatenate(true_batches, axis=0)\n    y_pred = np.concatenate(prediction_batches, axis=0)\n    return {\n        "overall": forecasting_metrics(y_true, y_pred),\n        "by_horizon": metrics_by_horizon(y_true, y_pred),\n        "by_node_type": metrics_by_node_type(y_true, y_pred, data.node_type_ids.numpy()),\n        "y_true": y_true,\n        "y_pred": y_pred,\n        "target_indices": np.asarray(target_indices, dtype=np.int64),\n        "inference_seconds": inference_seconds,\n    }\n\n\ndef train_model(model_name: str, config: Dict[str, Any], project_root: Path, resume: Path | None = None) -> Dict[str, Any]:\n    """Validation-only training. Test evaluation is an explicit later stage.\n\n    Interrupted attempts recover from the last fully committed epoch. Any\n    partial epoch is replayed in an explicit, budget-counted retry.\n    """\n    if not config.get("run", {}).get("experiment_id"):\n        raise RuntimeError("Training must be launched by the budget controller with a registered run")\n    set_seed(int(config["seed"]), bool(config.get("deterministic", True)))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    data = ExperimentData(config, project_root)\n    loaders = data.loaders(int(config["training"]["batch_size"]))\n    model = build_model(model_name, config, data).to(device)\n    training_config = config["training"]\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=float(training_config["learning_rate"]),\n        weight_decay=float(training_config["weight_decay"]),\n    )\n    scheduler = ReduceLROnPlateau(\n        optimizer,\n        mode="min",\n        factor=float(training_config["scheduler_factor"]),\n        patience=int(training_config["scheduler_patience"]),\n    )\n    epochs = int(training_config["epochs"])\n    patience = int(training_config["patience"])\n    clip_norm = float(training_config["gradient_clip_norm"])\n    delta = float(training_config.get("huber_delta", 1.0))\n    best_validation = float("inf")\n    best_state = None\n    wait = 0\n    history = []\n    start_epoch = 1\n    prior_seconds = 0.0\n    outputs = config["outputs"]\n    checkpoint_dir = project_root / outputs["checkpoint_dir"]\n    metrics_dir = project_root / outputs["metrics_dir"]\n    predictions_dir = project_root / outputs["predictions_dir"]\n    logs_dir = project_root / outputs["logs_dir"]\n    for directory in (checkpoint_dir, metrics_dir, predictions_dir, logs_dir):\n        directory.mkdir(parents=True, exist_ok=True)\n    latest_path = checkpoint_dir / "latest.pt"\n    best_path = checkpoint_dir / "best.pt"\n    run = config["run"]\n    metadata = {\n        "model_name": model_name,\n        "configuration": config,\n        "signature": run["signature"],\n        "node_names": data.node_names,\n        "node_type_ids": data.node_type_ids,\n        "edge_index": data.edge_index,\n        "scaler_state": data.scaler_state,\n        "split_ranges": data.ranges,\n        "dates": data.dates,\n    }\n    if resume is not None:\n        saved = torch.load(resume, map_location="cpu", weights_only=False)\n        if saved["signature"] != run["signature"] or saved["node_names"] != data.node_names:\n            raise ValueError("Resume checkpoint does not match code/config/dataset signature or node order")\n        model.load_state_dict(saved["state_dict"])\n        optimizer.load_state_dict(saved["optimizer"])\n        scheduler.load_state_dict(saved["scheduler"])\n        best_state = saved["best_state"]\n        best_validation = saved["best_validation_loss"]\n        wait = saved["wait"]\n        history = saved["history"]\n        start_epoch = int(saved["epoch"]) + 1\n        prior_seconds = float(saved["training_seconds"])\n        restore_rng_state(saved["rng_state"])\n        atomic_torch_save({**metadata, "state_dict": best_state}, best_path)\n        # A retry finishing after the last epoch still needs its own durable\n        # latest checkpoint and history, rather than pointing only at an old attempt.\n        atomic_torch_save({**saved, **metadata}, latest_path)\n        save_json({"history": history}, logs_dir / "history.json")\n        print(f"Resuming after committed epoch {saved[\'epoch\']}; this retry consumes another budget slot", flush=True)\n    start_time = time.perf_counter()\n    runtime_limit = float(config["experiment"]["max_job_seconds"])\n    edge_index = data.edge_index.to(device)\n    types = data.node_type_ids.to(device)\n\n    def check_time():\n        if time.perf_counter() - start_time >= runtime_limit:\n            raise TimeoutError("Training runtime limit reached; committed epoch checkpoints are preserved")\n\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(device)\n\n    print(f"Training {model_name} on {device}: nodes={data.num_nodes}, steps={data.num_steps}, "\n          f"train_windows={len(data.datasets[\'train\'])}, val_windows={len(data.datasets[\'validation\'])}, "\n          f"epochs={epochs}, batch_size={training_config[\'batch_size\']}", flush=True)\n\n    for epoch in range(start_epoch, epochs + 1):\n        if wait >= patience:\n            break\n        check_time()\n        model.train()\n        running_loss = 0.0\n        sample_count = 0\n        epoch_started = time.perf_counter()\n        for step, batch in enumerate(loaders["train"], start=1):\n            check_time()\n            batch = {key: value.to(device) for key, value in batch.items()}\n            optimizer.zero_grad(set_to_none=True)\n            prediction = _call_model(model_name, model, batch, edge_index, types)\n            loss = F.huber_loss(prediction, batch["y"], delta=delta)\n            if not torch.isfinite(loss):\n                raise ValueError("Nonfinite training loss")\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)\n            optimizer.step()\n            count = batch["x"].shape[0]\n            running_loss += float(loss.detach().cpu()) * count\n            sample_count += count\n            if step == 1 or step % 25 == 0 or step == len(loaders["train"]):\n                print(f"[{model_name}] Epoch {epoch}/{epochs} | batch {step}/{len(loaders[\'train\'])} "\n                      f"| mean_train_loss={running_loss / sample_count:.6f}", flush=True)\n        train_loss = running_loss / sample_count\n\n        model.eval()\n        validation_loss = 0.0\n        validation_count = 0\n        with torch.no_grad():\n            for batch in loaders["validation"]:\n                check_time()\n                batch = {key: value.to(device) for key, value in batch.items()}\n                prediction = _call_model(model_name, model, batch, edge_index, types)\n                count = batch["x"].shape[0]\n                validation_loss += float(F.huber_loss(prediction, batch["y"], delta=delta).cpu()) * count\n                validation_count += count\n        validation_loss /= validation_count\n        if not np.isfinite(validation_loss):\n            raise ValueError("Nonfinite validation loss")\n        scheduler.step(validation_loss)\n        history.append(\n            {\n                "epoch": epoch,\n                "train_loss": train_loss,\n                "validation_loss": validation_loss,\n                "learning_rate": optimizer.param_groups[0]["lr"],\n            }\n        )\n        if validation_loss < best_validation:\n            best_validation = validation_loss\n            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n            atomic_torch_save({**metadata, "state_dict": best_state}, best_path)\n            wait = 0\n        else:\n            wait += 1\n        latest = {\n            **metadata, "state_dict": model.state_dict(), "epoch": epoch,\n            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),\n            "best_state": best_state, "best_validation_loss": best_validation,\n            "wait": wait, "history": history, "rng_state": capture_rng_state(),\n            "training_seconds": prior_seconds + time.perf_counter() - start_time,\n        }\n        atomic_torch_save(latest, latest_path)\n        save_json({"history": history}, logs_dir / "history.json")\n        print(f"[{model_name}] Epoch {epoch}/{epochs} complete | train={train_loss:.6f} "\n              f"| val={validation_loss:.6f} | best_val={best_validation:.6f} "\n              f"| patience={wait}/{patience} | lr={optimizer.param_groups[0][\'lr\']:.2g} "\n              f"| elapsed={time.perf_counter() - epoch_started:.1f}s | checkpoint={latest_path}", flush=True)\n    if best_state is None:\n        raise RuntimeError("No checkpoint was selected")\n    model.load_state_dict(best_state)\n    training_seconds = prior_seconds + time.perf_counter() - start_time\n    evaluation = evaluate_loader(model_name, model, loaders["validation"], data, device)\n    inference_seconds = evaluation["inference_seconds"]\n    peak_memory = torch.cuda.max_memory_allocated(device) if device.type == "cuda" else 0\n\n    np.savez_compressed(predictions_dir / "validation.npz", y_true=evaluation["y_true"], y_pred=evaluation["y_pred"],\n                        target_indices=evaluation["target_indices"], node_names=data.node_names, dates=data.dates)\n    report = {\n        "model": model_name,\n        "device": str(device),\n        "parameter_count": model_parameter_count(model),\n        "training_seconds": training_seconds,\n        "inference_seconds": inference_seconds,\n        "peak_gpu_memory_bytes": int(peak_memory),\n        "epochs_completed": len(history),\n        "best_validation_loss": best_validation,\n        "selected_epoch": min(history, key=lambda item: item["validation_loss"])["epoch"],\n        "experiment_id": run["experiment_id"],\n        "signature": run["signature"],\n        "git_commit": run["git_commit"],\n        "validation": evaluation["overall"],\n        "validation_by_horizon": evaluation["by_horizon"],\n        "validation_by_node_type": evaluation["by_node_type"],\n        "checkpoint": str(best_path),\n        "latest_checkpoint": str(latest_path),\n    }\n    save_json(report, metrics_dir / "validation.json")\n    return report\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, default=Path("configs/experiment.yaml"))\n    parser.add_argument("--model", choices=["stgt", "stgt_permuted", "adaptive_stgt", *DAY2_VARIANTS], required=True)\n    parser.add_argument("--project-root", type=Path, default=Path("."))\n    parser.add_argument("--resume", type=Path, default=None)\n    parser.add_argument("--max-steps", "--max-time-steps", type=int, default=None,\n                        help="Optional chronological truncation of time steps (e.g. 221)")\n    return parser\n\n\nif __name__ == "__main__":\n    arguments = build_parser().parse_args()\n    root = arguments.project_root.resolve()\n    configuration = load_yaml(root / arguments.config)\n    if arguments.max_steps is not None:\n        configuration.setdefault("data", {})["max_time_steps"] = arguments.max_steps\n    try:\n        report = train_model(arguments.model, configuration, root, arguments.resume)\n        print(report)\n    except TimeoutError as exc:\n        print(f"Training time limit reached: {exc}; latest committed checkpoint is preserved", flush=True)\n        raise SystemExit(124)\n\n',
}
CONFIG_TEXT = 'dataset: isomorph_t221\nmax_time_steps: 221\nseed: 42\nseeds:\n- 42\n- 123\n- 2026\n- 7\n- 99\ndeterministic: true\n\nexperiment:\n  name: isomorph_t221_control\n  allow_execution: true\n  max_gpu_runs: 20\n  max_job_seconds: 7200\n  max_retries_per_signature: 0\n  one_active_run: true\n  overall_max_seconds: 28800\n  registry_path: results/isomorph_t221/experiment_registry_isomorph_t221.json\n  require_archive_validation: false\n  require_drive_authorization: false\n  require_pushed_commit: false\n\ndata:\n  processed_dir: data/isomorph\n  scaler: standard\n  train_ratio: 0.70\n  validation_ratio: 0.15\n  lookback: 14\n  horizon: 7\n  undirected_graph: false\n  use_residual_baseline: false\n  graph_edge_types:\n  - directed_route\n  max_time_steps: 221\n\ntraining:\n  batch_size: 4\n  epochs: 100\n  patience: 15\n  learning_rate: 0.001\n  weight_decay: 0.0001\n  gradient_clip_norm: 1.0\n  loss: huber\n  huber_delta: 1.0\n  scheduler_factor: 0.5\n  scheduler_patience: 8\n\nmodels:\n  temporal_only:\n    d_model: 64\n    temporal_heads: 2\n    spatial_heads: 4\n    dropout: 0.30\n  stgt:\n    d_model: 64\n    temporal_heads: 2\n    spatial_heads: 4\n    dropout: 0.30\n  stgt_permuted:\n    d_model: 64\n    temporal_heads: 2\n    spatial_heads: 4\n    dropout: 0.30\n  adaptive_stgt:\n    d_model: 64\n    temporal_heads: 2\n    spatial_heads: 4\n    dropout: 0.30\n\noutputs:\n  checkpoint_dir: results/isomorph_t221/checkpoints\n  metrics_dir: results/isomorph_t221/metrics\n  predictions_dir: results/isomorph_t221/predictions\n  logs_dir: results/isomorph_t221/logs\n  drive_artifact_root: results/isomorph_t221\n'

SUPPORT.mkdir(parents=True, exist_ok=True)
(REPO / 'src').mkdir(parents=True, exist_ok=True)
(REPO / 'configs').mkdir(parents=True, exist_ok=True)

for name, contents in HELPERS.items():
    (SUPPORT / name).write_text(contents, encoding='utf-8')
    (REPO / 'src' / name).write_text(contents, encoding='utf-8')

(REPO / 'configs/isomorph_t221_control.yaml').write_text(CONFIG_TEXT, encoding='utf-8')

# Ensure latest code modules are loaded fresh
for mod in ['utils', 'train', 'kaggle_drive', 'kaggle_control_campaign']:
    if mod in sys.modules:
        del sys.modules[mod]

sys.path.insert(0, str(SUPPORT))
sys.path.insert(0, str(REPO / 'src'))

from kaggle_drive import DriveStore, get_drive, account_email, find_token_file

token_path = find_token_file(TOKEN_FILE)
print('Resolved Drive token path:', token_path)
drive = get_drive(token_path)
print('Authenticated Google Drive account:', account_email(drive))

root_store = DriveStore(drive, DRIVE_ROOT_FOLDER_ID)
ISOMORPH_PROCESSED.mkdir(parents=True, exist_ok=True)

# Check for pre-attached Kaggle Dataset first
kaggle_dataset_path = Path('/kaggle/input/datasets/mohdakifaliparvez/isomorph-sample-processed/processed_inventory')
if kaggle_dataset_path.exists():
    print(f'Found pre-attached Kaggle dataset at {kaggle_dataset_path}. Copying...')
    import shutil
    for name in ['nodes.csv', 'edges.csv', 'sales.csv', 'target_definition.json', 'data_quality.json']:
        src = kaggle_dataset_path / name
        if src.exists():
            shutil.copy2(src, ISOMORPH_PROCESSED / name)
else:
    print('Downloading ISOMORPH processed data from Google Drive...')
    source_folder = 'experiments/isomorph_sample_kaggle_campaign/processed_inventory'
    for name in ['nodes.csv', 'edges.csv', 'sales.csv', 'target_definition.json', 'data_quality.json']:
        dest = ISOMORPH_PROCESSED / name
        if not dest.exists():
            root_store.download(f'{source_folder}/{name}', dest)

present_files = sorted(f.name for f in ISOMORPH_PROCESSED.glob('*') if f.is_file())
print('Downloaded processed data:', ISOMORPH_PROCESSED)
print('Files present:', present_files)
assert 'nodes.csv' in present_files and 'edges.csv' in present_files and 'sales.csv' in present_files, 'Missing required files!'

In [ ]:
# CELL 4: Sanity Checks (T=221 Slice, Permuted Graph, Active Node Variance)
import numpy as np
import pandas as pd
import torch

try:
    from utils import read_processed_data, build_edge_index, split_target_ranges, fit_scale_matrix, permute_edge_index
except ImportError:
    from utils import read_processed_data, build_edge_index, split_target_ranges, fit_scale_matrix
    def permute_edge_index(edge_index: torch.Tensor, num_nodes: int, graph_seed: int = 0) -> torch.Tensor:
        real_pairs = set(zip(edge_index[0].tolist(), edge_index[1].tolist()))
        num_edges = edge_index.shape[1]
        rng = np.random.default_rng(graph_seed)
        for _ in range(1000):
            p = rng.permutation(num_nodes)
            candidate = [(int(p[u]), int(p[v])) for u, v in zip(edge_index[0].tolist(), edge_index[1].tolist())]
            if any(u == v for u, v in candidate) or len(set(candidate)) != num_edges or any(pair in real_pairs for pair in candidate):
                continue
            return torch.tensor(sorted(candidate), dtype=torch.long).t().contiguous()
        all_possible = [(u, v) for u in range(num_nodes) for v in range(num_nodes) if u != v and (u, v) not in real_pairs]
        chosen = sorted([all_possible[i] for i in rng.choice(len(all_possible), size=num_edges, replace=False)])
        return torch.tensor(chosen, dtype=torch.long).t().contiguous()

nodes, edges, sales = read_processed_data(ISOMORPH_PROCESSED)
node_names = nodes['node_id'].astype(str).tolist()
num_nodes = len(node_names)
assert num_nodes == 13, f'Expected 13 nodes, found {num_nodes}'
print(f'[Sanity Check 1 Passed] ISOMORPH has {num_nodes} nodes: {node_names}')

# Real directed routing edge index
edge_index = build_edge_index(nodes, edges, undirected=False)
assert edge_index.shape[1] == 16, f'Expected 16 directed edges, found {edge_index.shape[1]}'
print(f'[Sanity Check 2 Passed] Real directed edge index has {edge_index.shape[1]} edges.')

# Permuted edge index with graph_seed=0
perm_index = permute_edge_index(edge_index, num_nodes, graph_seed=0)
real_pairs = set(zip(edge_index[0].tolist(), edge_index[1].tolist()))
perm_pairs = set(zip(perm_index[0].tolist(), perm_index[1].tolist()))
shared = len(real_pairs.intersection(perm_pairs))
assert len(perm_pairs) == 16, f'Expected 16 permuted edges, found {len(perm_pairs)}'
assert shared == 0, f'Permuted graph must have exactly 0 shared edges with real routing graph, found {shared}'
print(f'[Sanity Check 3 Passed] Permuted graph (seed 0) has 16 edges and 0 shared edges with real graph.')

# Sliced T=221 series
raw_matrix = sales[node_names].to_numpy(dtype=np.float32).T
T_TARGET = 221
assert raw_matrix.shape[1] >= T_TARGET, f'Insufficient series steps: {raw_matrix.shape[1]} < {T_TARGET}'
sliced_matrix = raw_matrix[:, :T_TARGET]
print(f'[Sanity Check 4 Passed] Truncated raw series to T={T_TARGET}. Shape: {sliced_matrix.shape}')

# Active non-constant node variance check
node_vars = np.var(sliced_matrix, axis=1)
zero_var = [node_names[i] for i, v in enumerate(node_vars) if v <= 0.0]
assert len(zero_var) == 0, f'Zero-variance nodes detected in T=221 slice: {zero_var}'
print(f'[Sanity Check 5 Passed] All 13 nodes exhibit active non-constant variance (min var = {np.min(node_vars):.4f}).')

# Windows check
ranges = split_target_ranges(T_TARGET, lookback=14, horizon=7, train_ratio=0.70, validation_ratio=0.15)
train_w = ranges['train'][1] - ranges['train'][0]
val_w = ranges['validation'][1] - ranges['validation'][0]
test_w = ranges['test'][1] - ranges['test'][0]
assert train_w == 134 and val_w == 27 and test_w == 28, f'Unexpected window counts: train={train_w}, val={val_w}, test={test_w}'
print(f'[Sanity Check 6 Passed] Chronological Windows: Train={train_w} (134), Val={val_w} (27), Test={test_w} (28, matches SupplyGraph Table I).')

In [ ]:
# CELL 5: Execution Loop with Automatic Exit Code 124 Resumption
import subprocess
import sys
import pandas as pd

campaign_command = [
    sys.executable, str(REPO / 'src/kaggle_control_campaign.py'),
    '--project-root', str(REPO),
    '--config', str(REPO / 'configs/isomorph_t221_control.yaml'),
    '--processed-root', str(ISOMORPH_PROCESSED),
    '--token-file', str(token_path),
    '--drive-root-folder-id', DRIVE_ROOT_FOLDER_ID,
    '--work-root', str(WORK_ROOT),
    '--campaign', 'isomorph_t221_control',
    '--dataset', 'isomorph',
    '--models', 'temporal_only', 'stgt', 'stgt_permuted', 'adaptive_stgt',
    '--seeds', '42', '123', '2026', '7', '99',
    '--max-steps', '221',
    '--results-name', 'isomorph_t221_results.csv',
    '--allow-execution',
]

print('Launching ISOMORPH T=221 Control Campaign...')
print('Command:', ' '.join(campaign_command))

while True:
    result = subprocess.run(campaign_command, cwd=REPO)
    if result.returncode == 0:
        print('\n>>> ISOMORPH T=221 Control Campaign completed all 20 runs successfully! <<<')
        break
    elif result.returncode == 124:
        print('\n>>> Checkpoint boundary / timeout reached (exit code 124); automatically resuming from Drive checkpoint... <<<')
        continue
    else:
        raise RuntimeError(f'Campaign stopped unexpectedly with exit code {result.returncode}')

In [ ]:
# CELL 6: Statistical Analysis & Hypothesis Evaluation (df=4)
import pandas as pd
import numpy as np
from scipy import stats

csv_path = WORK_ROOT / 'isomorph_t221_results.csv'
if not csv_path.exists():
    # Attempt download from Drive if not found locally
    campaign_folder_id = root_store.folder('experiments/isomorph_t221_control')
    camp_store = DriveStore(drive, campaign_folder_id)
    camp_store.download('isomorph_t221_results.csv', csv_path)

df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} completed run results:')
print(df[['model', 'seed', 'MAE', 'RMSE', 'WAPE', 'R2', 'epochs_completed', 'training_seconds']])

# Group statistics
models = ['temporal_only', 'stgt', 'stgt_permuted', 'adaptive_stgt']
summary_rows = []
for m in models:
    sub = df[df['model'] == m]
    if len(sub) > 0:
        summary_rows.append({
            'Model': m,
            'Runs': len(sub),
            'MAE Mean': sub['MAE'].mean(),
            'MAE Std': sub['MAE'].std(),
            'RMSE Mean': sub['RMSE'].mean(),
            'RMSE Std': sub['RMSE'].std(),
            'WAPE Mean': sub['WAPE'].mean(),
            'WAPE Std': sub['WAPE'].std(),
            'R2 Mean': sub['R2'].mean(),
            'R2 Std': sub['R2'].std(),
        })

sum_df = pd.DataFrame(summary_rows)
print('\n' + '='*80)
print('ISOMORPH T=221 CONTROL SUMMARY (5 Seeds: 42, 123, 2026, 7, 99)')
print('='*80)
print(sum_df.to_string(index=False))

# Paired t-tests across seeds
print('\n' + '='*80)
print('PAIRED T-TESTS (df = 4, Two-Sided)')
print('='*80)

def paired_test(m1, m2):
    s1 = df[df['model'] == m1].sort_values('seed')['MAE'].to_numpy()
    s2 = df[df['model'] == m2].sort_values('seed')['MAE'].to_numpy()
    if len(s1) == 5 and len(s2) == 5:
        diff = s1 - s2
        t_stat, p_val = stats.ttest_rel(s1, s2)
        print(f'{m1} vs {m2}:')
        print(f'  MAE Diff (m1 - m2): {diff.mean():.6f} ± {diff.std(ddof=1):.6f}')
        print(f'  t-statistic: {t_stat:.4f}, p-value: {p_val:.4e} (df = 4)')
        if p_val < 0.01:
            print('  -> Statistically significant at p < 0.01 (***)')
        elif p_val < 0.05:
            print('  -> Statistically significant at p < 0.05 (**)')
        else:
            print('  -> Not statistically significant (p >= 0.05)')
    else:
        print(f'Insufficient runs for paired test: {m1} ({len(s1)}), {m2} ({len(s2)})')

paired_test('stgt', 'stgt_permuted')
paired_test('temporal_only', 'stgt')
paired_test('temporal_only', 'stgt_permuted')
paired_test('stgt', 'adaptive_stgt')